# CatBoost

CatBoost (short for Categorical Boosting) is a high-performance, open-source gradient boosting library developed by Yandex that builds sequential decision trees to handle classification and regression tasks

# more details click the below link 

https://docs.google.com/document/d/1CQe8H9m46Kp6njs_rCAquZDc4N6Nag7xBnHfEVolmIg/edit?usp=sharing

# notes and question and answer. 

https://docs.google.com/document/d/114ImM2Pv5ymrlt_v4EG1pJucszmVhjyrgMbmjUj_Sgs/edit?usp=sharing

# Coding

# Import libraries

In [1]:
! pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 7.2 MB/s  0:00:04m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 5.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [catboost]2/3 [catboost]

[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np                              # Numerical operations
import pandas as pd                             # Data handling

from sklearn.datasets import load_breast_cancer
# Loads the prepared Breast Cancer dataset

from sklearn.model_selection import train_test_split
# Splits the data into training and testing sets

from sklearn.metrics import (
    accuracy_score,                             # Calculates accuracy
    confusion_matrix,                           # Creates confusion matrix
    classification_report                       # Precision, recall, F1-score
)

from catboost import CatBoostClassifier
# Imports CatBoost classification algorithm

# Load the dataset 

In [3]:
data = load_breast_cancer()
# Loads the Breast Cancer dataset

In [4]:
print("Dataset Shape:", data.data.shape)
# Shows number of samples and number of features

Dataset Shape: (569, 30)


In [9]:
# spreate X and y 
X = data.data
# X contains the 30 input features

y = data.target
# y contains the target labels

print("Sample:-\n", X)


Sample:-
 [[1.799e+01 1.038e+01 1.228e+02 ... 2.654e-01 4.601e-01 1.189e-01]
 [2.057e+01 1.777e+01 1.329e+02 ... 1.860e-01 2.750e-01 8.902e-02]
 [1.969e+01 2.125e+01 1.300e+02 ... 2.430e-01 3.613e-01 8.758e-02]
 ...
 [1.660e+01 2.808e+01 1.083e+02 ... 1.418e-01 2.218e-01 7.820e-02]
 [2.060e+01 2.933e+01 1.401e+02 ... 2.650e-01 4.087e-01 1.240e-01]
 [7.760e+00 2.454e+01 4.792e+01 ... 0.000e+00 2.871e-01 7.039e-02]]


# Train Test Split 

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Input features
    y,                  # Target labels
    test_size=0.2,      # 20% data goes to testing
    stratify=y,         # Keeps class proportions similar
    random_state=42     # Makes the split reproducible
)

In [17]:
# create the baseline CatBoost Model 
classifier = CatBoostClassifier(
    random_state=42,
    verbose=0
)
# Creates the CatBoost classification model

In [20]:
# Train the model 

classifier.fit(X_train, y_train) # now catboost learned patterns from my training data. 


CatBoostClassifier(random_state=42, verbose=0)

In [21]:
# training prediction 

X_train_prediction  = classifier.predict(X_train)

# predict the training samples

# Calculate accracy 
training_data_accuracy = accuracy_score(y_train, X_train_prediction)

# Compare actual training labels with predicted labels 

print("Training Accuracy:", training_data_accuracy)

Training Accuracy: 1.0


# Test prediction

In [22]:
X_test_prediction = classifier.predict(
    X_test
)
# Predicts unseen test samples
# accuracy score

test_data_accuracy = accuracy_score(
    y_test,
    X_test_prediction
)
# Compares actual test labels with predicted labels

print(
    "Test Accuracy:",
    test_data_accuracy
)


Test Accuracy: 0.956140350877193


In [23]:
# accuracy gap 

accuracy_gap = (
    training_data_accuracy
    - test_data_accuracy
)
# Measures the difference between training and test accuracy

print(
    "Accuracy Gap:",
    accuracy_gap
)

Accuracy Gap: 0.04385964912280704


In [24]:
# confusion matrix 

conf_matrix = confusion_matrix(
    y_test,
    X_test_prediction
)
# Shows correct and incorrect predictions

print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[38  4]
 [ 1 71]]


In [25]:
print(
    classification_report(
        y_test,
        X_test_prediction,
        target_names=data.target_names
    )
)
# Displays precision, recall, F1-score and support

              precision    recall  f1-score   support

   malignant       0.97      0.90      0.94        42
      benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



# Our simple training model is prety good not bad yet . 

# But now we gonna experiment with so many parameters. 

# Experiment - iterations 
`In CatBoost, iterations is roughly the number of boosting rounds/trees.`


In [26]:
iterations_values = [
    30,
    50,
    100,
    150,
    200
]


results = []

for n in iterations_values:

    model = CatBoostClassifier(
        iterations=n,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=0
    )

    # Train the model
    model.fit(
        X_train,
        y_train
    )

    # Training prediction
    train_prediction = model.predict(
        X_train
    )

    # Test prediction
    test_prediction = model.predict(
        X_test
    )

    # Calculate training accuracy
    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    # Calculate test accuracy
    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    # Calculate accuracy gap
    gap = train_accuracy - test_accuracy

    results.append({
        "Iterations": n,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": gap
    })


iterations_results = pd.DataFrame(results)

print(iterations_results)

   Iterations  Train Accuracy  Test Accuracy  Accuracy Gap
0          30        0.995604       0.956140      0.039464
1          50        1.000000       0.956140      0.043860
2         100        1.000000       0.956140      0.043860
3         150        1.000000       0.964912      0.035088
4         200        1.000000       0.964912      0.035088


# Experiment - learning rate 


In [29]:
learning_rate_values = [
    0.01,
    0.05,
    0.1,
    0.2
]
results = []

for lr in learning_rate_values:

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=lr,
        depth=6,
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    train_prediction = model.predict(
        X_train
    )

    test_prediction = model.predict(
        X_test
    )

    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    gap = train_accuracy - test_accuracy

    results.append({
        "Learning Rate": lr,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": gap
    })


learning_rate_results = pd.DataFrame(results)

print(learning_rate_results)

   Learning Rate  Train Accuracy  Test Accuracy  Accuracy Gap
0           0.01        0.986813       0.947368      0.039445
1           0.05        1.000000       0.956140      0.043860
2           0.10        1.000000       0.956140      0.043860
3           0.20        1.000000       0.947368      0.052632


# Experiment - depth 

`Depth controls the depth of the individual decisio trees . `

In [31]:
depth_values = [
    3,
    4,
    5,
    6,
    8,
    10
]

results = []

for d in depth_values:

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=d,
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    train_prediction = model.predict(
        X_train
    )

    test_prediction = model.predict(
        X_test
    )

    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    gap = train_accuracy - test_accuracy

    results.append({
        "Depth": d,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": gap
    })


depth_results = pd.DataFrame(results)

print(depth_results)


   Depth  Train Accuracy  Test Accuracy  Accuracy Gap
0      3             1.0       0.964912      0.035088
1      4             1.0       0.947368      0.052632
2      5             1.0       0.947368      0.052632
3      6             1.0       0.956140      0.043860
4      8             1.0       0.964912      0.035088
5     10             1.0       0.964912      0.035088


# Experiment - l2_lear_reg

`This is CatBoost L2 regularization parameter.`

In [32]:
l2_values = [
    1,
    3,
    5,
    10,
    20
]


results = []

for reg in l2_values:

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        l2_leaf_reg=reg,
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    train_prediction = model.predict(
        X_train
    )

    test_prediction = model.predict(
        X_test
    )

    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    gap = train_accuracy - test_accuracy

    results.append({
        "L2 Leaf Reg": reg,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": gap
    })


l2_results = pd.DataFrame(results)

print(l2_results)

   L2 Leaf Reg  Train Accuracy  Test Accuracy  Accuracy Gap
0            1             1.0       0.947368      0.052632
1            3             1.0       0.956140      0.043860
2            5             1.0       0.947368      0.052632
3           10             1.0       0.956140      0.043860
4           20             1.0       0.947368      0.052632


# Experiment with - random_strength

# CatBoost can add randomness when choosing tree splits . 

In [34]:
random_strength_values = [
    0,
    0.5,
    1,
    2,
    5
]
results = []

for strength in random_strength_values:

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=6,
        random_strength=strength,
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train,
        y_train
    )

    train_prediction = model.predict(
        X_train
    )

    test_prediction = model.predict(
        X_test
    )

    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    gap = train_accuracy - test_accuracy

    results.append({
        "Random Strength": strength,
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "Accuracy Gap": gap
    })


random_strength_results = pd.DataFrame(results)

print(random_strength_results)


   Random Strength  Train Accuracy  Test Accuracy  Accuracy Gap
0              0.0             1.0       0.956140      0.043860
1              0.5             1.0       0.956140      0.043860
2              1.0             1.0       0.956140      0.043860
3              2.0             1.0       0.956140      0.043860
4              5.0             1.0       0.964912      0.035088


# Feature Importance 

`CatBoost can tell us which features contributed. most to the model`

In [36]:
feature_importance = pd.DataFrame({
    "Feature": data.feature_names,
    "Importance": classifier.get_feature_importance()
})
# Creates a table containing feature names and their importance

# Select Final CatBoost Model 

`After looking at your experiment results, choose your parameters.`

In [38]:
# Don't copy this blindly chose from your best predict model score 

final_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=5,
    l2_leaf_reg=3,
    random_strength=1,
    random_seed=42,
    verbose=0
)

In [39]:
# Train Final model 

final_model.fit(
    X_train,
    y_train
)
# Trains the selected final CatBoost model

CatBoostClassifier(depth=5, iterations=100, l2_leaf_reg=3, learning_rate=0.1, random_seed=42, random_strength=1, verbose=0)

In [40]:
# Final predictions 

final_train_prediction = final_model.predict(
    X_train
)
# Predict training data

final_test_prediction = final_model.predict(
    X_test
)
# Predict unseen test data

In [41]:
# Final Evaluation 
final_train_prediction = final_model.predict(
    X_train
)
# Predict training data

final_test_prediction = final_model.predict(
    X_test
)
# Predict unseen test data


In [42]:
# Confusion matrix 
print("Final Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        final_test_prediction
    )
)

Final Confusion Matrix:
[[38  4]
 [ 2 70]]


In [43]:
# Final classification model 

print("Final Classification Report:")

print(
    classification_report(
        y_test,
        final_test_prediction,
        target_names=data.target_names
    )
)

Final Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.90      0.93        42
      benign       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



In [44]:
# Save the model 

import pickle

# Save the trained CatBoost model
with open(
    "catboost_breast_cancer.pkl",
    "wb"
) as file:

    pickle.dump(
        final_model,
        file
    )

print("CatBoost model saved successfully!")

CatBoost model saved successfully!


In [45]:
# load the saved model 

with open(
    "catboost_breast_cancer.pkl",
    "rb"
) as file:

    loaded_model = pickle.load(file)

print("CatBoost model loaded successfully!")

CatBoost model loaded successfully!
